# 4.6 最小能量设计

图4.22展示了美国本土(不含阿拉斯加,夏威夷)各邮政编码区域的人口密度(罗齐,2022).图中粉色点代表各个区域,点的大小与人口密度的对数值成正比.假设我们需要布设200个传感器来测量全美污染水平;获取传感器观测数据后,可以利用克里金法或高斯过程预测美国任意地点的污染浓度.应当如何布设传感器,才能保证预测精度?利用R软件包`maximin`生成的200点极大极小设计如图4.22左图所示,这种布设方案并不合理:我们知道人口稠密区域的污染水平通常更高.理想情况下,应当在人口高密度区域布设更多传感器,低密度区域布设更少传感器.因此,我们需要一类加权空间填充设计(鲍曼,伍兹,2013),权重与人口密度成正比.约瑟夫等人(2015)提出的最小能量设计(MED)正是这类方案之一.

> **图4.22** 极大极小设计点与最小能量设计点以蓝色圆点标出.2020年人口普查得到的美国各地邮政编码点位以粉色圆点呈现,圆点大小与人口密度成正比.

In [9]:
#图4.22

options(repr.plot.width=30,repr.plot.height=10)
par(mfrow=c(1,2))
library(zipcodeR)
dat=na.omit(zip_code_db[,c(9,8,14)])
dat=dat[(dat[,2]>25&dat[,2]<52),]
CAND=as.matrix(dat[,1:2])
y=dat[,3]
sizecode=.5+log(y+1)/max(log(y+1))

plot(CAND,cex=sizecode,pch=16,col="pink",xlab="",ylab="",main="极大极小设计",cex.main=4)
D0=CAND[sample(nrow(CAND),1),,drop=FALSE]
library(maximin)
D2=CAND[maximin.cand(199,Xcand=CAND,Xorig=D0)$inds,]
D2=rbind(D0,D2)
points(D2,pch=16,col="blue",cex=3)

plot(CAND,cex=sizecode,pch=16,col="pink",xlab="",ylab="",main="最小能量设计",cex.main=4)
library(mined)
D=SelectMinED(candidates=CAND,candlf=log(y),n=200,s=1)$points
points(D,pch=16,col="blue",cex=3)
par(mfrow=c(1,1))

Warning message in maximin.cand(199, Xcand = CAND, Xorig = D0):
"terminated early since the maximum progress has been achieved :-)"


最小能量设计存在直观的物理类比:考虑$n$个带电粒子,不妨设均带正电.若将粒子放置在一个盒子内部,粒子间相互排斥,最终会稳定在使总势能最小的位置.设粒子位于位置$\boldsymbol{x}\in\mathcal{X}=[0,1]^p$时携带电荷量为$q(\boldsymbol{x})$,则$n$个粒子的总势能为:

$$E=\sum_{i=1}^{n-1}\sum_{j=i+1}^{n}\frac{q(\boldsymbol{x}_i)q(\boldsymbol{x}_j)}{\|\boldsymbol{x}_i-\boldsymbol{x}_j\|}$$

最小能量设计通过关于设计$D=\{\boldsymbol{x}_1,\dots,\boldsymbol{x}_n\}$极小化势能$E$得到.若对所有$\boldsymbol{x}\in[0,1]^p$都满足$q(\boldsymbol{x})=1$,该准则就退化为式(4.18)中$\alpha=1/2$时的总倒数距离准则.由此可以给出能量准则的广义形式:

$$\min_{D}\left(\sum_{i=1}^{n-1}\sum_{j=i+1}^{n}\left(\frac{q(\boldsymbol{x}_i)q(\boldsymbol{x}_j)}{\|\boldsymbol{x}_i-\boldsymbol{x}_j\|_\alpha}\right)^k\right)^{1/k}$$

其中$\|\boldsymbol{x}\|_\alpha=\left\{\sum_{i=1}^p|x_i|^\alpha\right\}^{1/\alpha}$.当$k\to\infty$时,该准则收敛为:

$$\min_{D}\max_{i\ne j}\frac{q(\boldsymbol{x}_i)q(\boldsymbol{x}_j)}{\|\boldsymbol{x}_i-\boldsymbol{x}_j\|_\alpha}\equiv\max_{D}\min_{i\ne j}\frac{\|\boldsymbol{x}_i-\boldsymbol{x}_j\|_\alpha}{q(\boldsymbol{x}_i)q(\boldsymbol{x}_j)}\tag{4.43}$$

可以将其看作式(4.16)极大极小准则的加权版本.约瑟夫等人(2015,2019)证明了如下渐近结论:

> **定理4.4** 假设电荷函数$q(\cdot)$满足利普希茨连续,即存在常数$L>0$,使得对任意$\boldsymbol{u},\boldsymbol{v}\in\mathcal{X}=[0,1]^p$,有$|q(\boldsymbol{u})-q(\boldsymbol{v})|\le L\mathrm{d}(\boldsymbol{u},\boldsymbol{v})$.设$D^*=\{\boldsymbol{x}_1^*,\dots,\boldsymbol{x}_n^*\}$是采用准则(4.43),选取最小指标得到的$n$点最小能量设计,$\mathscr{B}$为$\mathcal{X}$上的博雷尔$\sigma$代数.定义$(\mathcal{X},\mathscr{B})$上的概率测度:$$\mathcal{P}_n(A)=\frac{\mathrm{card}\{\boldsymbol{x}_i^*:1\le i\le n,\boldsymbol{x}_i^*\in A\}}{n},\;\forall A\in\mathscr{B}\tag{4.44}$$ 则存在概率测度$\mathcal{P}$,使得对任意固定$\alpha\in(0,\infty)$,当$n\to\infty$时,$\mathcal{P}_n$依分布收敛到$\mathcal{P}$;并且$\mathcal{P}$在$\mathcal{X}$上的密度与$1/q^{2p}(\boldsymbol{x})$成正比.

有趣的是,在球面填充相关文献中,最小能量设计也被称作极小里斯能量点.$\alpha=2$情形下与定理4.4类似的结论可参考博罗达乔夫等人(2008a,b).

定理4.4极具实用价值:若我们预先给定目标分布密度$f(\boldsymbol{x})$,只需选取电荷函数$q(\boldsymbol{x})=f(\boldsymbol{x})^{-1/(2p)}$,对应的最小能量设计就会渐近服从目标分布.采用该电荷函数后,最小能量设计准则化为:

$$\max_{D}\min_{i\ne j}f^{1/(2p)}(\boldsymbol{x}_i)f^{1/(2p)}(\boldsymbol{x}_j)\|\boldsymbol{x}_i-\boldsymbol{x}_j\|_\alpha\tag{4.45}$$

最小能量设计具备清晰的概率解释.当$\alpha=2$时,准则(4.45)等价于:

$$\max_{D}\min_{i\ne j}\sqrt{f(\boldsymbol{x}_i)f(\boldsymbol{x}_j)}V_S(\boldsymbol{x}_i,\boldsymbol{x}_j)$$

其中$V_S(\boldsymbol{x}_i,\boldsymbol{x}_j)=\pi^{p/2}/\Gamma(p/2+1)(\mathrm{d}(\boldsymbol{x}_i,\boldsymbol{x}_j)/2)^p$代表以$(\boldsymbol{x}_i+\boldsymbol{x}_j)/2$为球心,同时经过$\boldsymbol{x}_i,\boldsymbol{x}_j$两点的超球体体积.项$\sqrt{f(\boldsymbol{x}_i)f(\boldsymbol{x}_j)}$是两点处密度的几何平均.因此$P_{ij}(D)=\sqrt{f(\boldsymbol{x}_i)f(\boldsymbol{x}_j)}V_S(\boldsymbol{x}_i,\boldsymbol{x}_j)$可以近似理解为随机点$\boldsymbol{x}$落入该超球内的概率.记$i^*=\argmin_{j\ne i}P_{ij}(D)$,则最小能量设计准则可以写作:

$$\max_{D}\min_{i=1:n}P_{ii^*}(D)$$

最大化最小概率,会促使所有$i=1,\dots,n$对应的$P_{ii^*}(D)$趋于相等.粗略来说,最小能量设计旨在平衡设计中相邻点对应的概率.如式(4.25)所示,当$\alpha=0$时,准则(4.45)等价于:

$$\max_{D}\min_{i\ne j}\sqrt{f(\boldsymbol{x}_i)f(\boldsymbol{x}_j)}V_R(\boldsymbol{x}_i,\boldsymbol{x}_j)$$

其中$V_R(\boldsymbol{x}_i,\boldsymbol{x}_j)=\prod_{l=1}^p|x_{il}-x_{jl}|$代表以$\boldsymbol{x}_i,\boldsymbol{x}_j$为对角顶点,各边平行于坐标轴的超长方体体积.该准则拥有和欧氏距离情形相同的"概率均衡"解释,只是将超球体体积替换为超长方体体积单元.当$f(\boldsymbol{x})=1$(均匀目标密度)时,$\alpha=2$对应极大极小设计,$\alpha=0$对应极大投影设计.

约瑟夫等人(2015)提出最小能量设计的序贯构造算法:

$$\boldsymbol{x}_{k+1}=\argmax_{\boldsymbol{x}\in\mathcal{X}}\min_{i=1:k}f(\boldsymbol{x})f(\boldsymbol{x}_i)\|\boldsymbol{x}-\boldsymbol{x}_i\|_\alpha^{2p},\;k=2,\dots,n\tag{4.46}$$

其中

$$\boldsymbol{x}_1=\arg\max_{\boldsymbol{x}\in\mathcal{X}}f(\boldsymbol{x})$$

再来回顾基于人口密度$f(\boldsymbol{x})$布设全美污染监测传感器的问题.式(4.46)中的序贯算法在R包`mined`中实现.取$\alpha=1$,通过最小能量设计选出的200个点位如图4.22右图所示.可以看到:人口聚居的邮政编码区域分配了更多传感器,有助于构建精度更高的全美污染水平预测模型.